# Review preprocessing and MiniLM embeddings

Organized from Zilan's notebook and Annie's working copy. This is the selected **cleaned, non-chunked** pipeline. It prepares inputs for `scripts/difficulty_model.py`; it does not train LogisticIT.

Run cells from top to bottom. Use the same private CSV and original row order. Keep exported data and notebook outputs private. This notebook removes duplicate cleaning cells, chunking experiments, and temporary debugging output. It preserves the January-header fix and both `random_state=42` splits. Long reviews use MiniLM's normal truncation behavior.

## 1. Install and import packages

In [ ]:
# install the tools in this Colab session; this cell is notebook-specific.
%pip install -q sentence-transformers scikit-learn pandas numpy

In [ ]:
# local command-line options are not needed in Colab.
from pathlib import Path
import re

# pandas reads the csv; numpy saves arrays; scikit-learn separates the review groups.
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split




## 2. Upload the private CSV

In [ ]:
# select the existing CSV from your computer; this does not scrape the website.
from google.colab import files
uploaded = files.upload()
csv_files = [name for name in uploaded if name.lower().endswith('.csv')]
if len(csv_files) != 1:
    raise ValueError('Upload exactly one review CSV.')
csv_path = Path(csv_files[0])
print('Using:', csv_path.name)

## 3. Define the cleaning rule

This is the accepted January-header correction. No additional transformations are introduced. Defining the function does not run it yet.

In [ ]:
def clean_review(text):
    """remove the same webpage headers and footers as the selected notebook version."""
    # the January option is the accepted correction; keep the rest of the rule unchanged.
    # this removes a term, year, and displayed average rating before the student's writing.
    text = re.sub(
        r'^(Fall|Spring|Summer|Winter|January)\s+\d{4}\s+\d+(?:\.\d+)?\s+AVERAGE\s*',
        '', text, flags=re.IGNORECASE
    )
    # this removes a trailing vote count and date, not sentences within the review.
    text = re.sub(
        r'\s+-?\d+\s+'
        r'(?:Jan\.?|Feb\.?|Mar\.?|Apr\.?|May|Jun\.?|Jul\.?|Aug\.?|'
        r'Sept?\.?|Oct\.?|Nov\.?|Dec\.?|January|February|March|April|'
        r'June|July|August|September|October|November|December)'
        r'\s+\d{1,2},\s+\d{4}\s*$',
        '', text, flags=re.IGNORECASE
    )
    return text.strip()




## 4. Clean the reviews and create the 70/15/15 split

The function keeps review text and labels aligned, preserving the original two stratified splits. Expected sizes for the selected CSV: 3,289 / 705 / 705.

In [ ]:
def prepare_splits(csv_path):
    """load, clean, and split the reviews without changing their original order."""
    reviews = pd.read_csv(csv_path)
    required = {"review_text", "difficulty_rating"}
    if not required.issubset(reviews.columns):
        raise ValueError("The CSV must contain review_text and difficulty_rating.")
    # fail on unexpected data instead of silently dropping rows and changing the saved split.
    # the selected dataset has no missing review texts or difficulty ratings.
    if reviews[list(required)].isna().any().any():
        raise ValueError("Missing reviews or ratings: resolve these before reproducing the split.")
    if not reviews["review_text"].map(lambda text: isinstance(text, str)).all():
        raise ValueError("Every review must be text.")
    if not reviews["difficulty_rating"].isin([1, 2, 3, 4, 5]).all():
        raise ValueError("Difficulty ratings must be whole numbers from 1 to 5.")

    # apply one definitive cleaning function, rather than cleaning twice with different rules.
    # no extra lowercase, punctuation, or markdown transformations are added to this selected version.
    reviews["cleaned_review"] = reviews["review_text"].apply(clean_review)
    if reviews["cleaned_review"].eq("").any():
        raise ValueError("Cleaning produced an empty review; inspect the input first.")
    X = reviews["cleaned_review"]
    y = reviews["difficulty_rating"]

    # reserve 30% temporarily, leaving 70% for training.
    # stratify preserves approximate rating proportions; 42 makes the selection repeatable.
    X_train, X_temp, y_train, y_temp = train_test_split(
        X, y, test_size=0.30, stratify=y, random_state=42
    )
    # divide the remaining 30% into equal validation and test groups (15% each overall).
    X_val, X_test, y_val, y_test = train_test_split(
        X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=42
    )
    splits = {"train": (X_train, y_train), "val": (X_val, y_val), "test": (X_test, y_test)}
    for name, (texts, labels) in splits.items():
        if not texts.index.equals(labels.index):
            raise ValueError(f"Review and label indexes do not match for {name}.")
        print(f"{name}: {len(texts)} reviews")
        print(labels.value_counts().sort_index().to_string())
    return splits




In [ ]:
# run the cleaning and splitting steps, printing only aggregate counts.
splits = prepare_splits(csv_path)

# these names retain the familiar text/label pairs from the original notebook.
X_train, y_train = splits['train']
X_val, y_val = splits['val']
X_test, y_test = splits['test']

## 5. Generate non-chunked embeddings and save the export

All three groups use the same direct MiniLM encode call. No ratings are passed to MiniLM. This may take several minutes. Package versions or hardware can cause numerical differences from earlier exports; preserve the existing selected export.

In [ ]:
def save_embeddings(splits, output_path):
    """use the pretrained model to encode whole reviews, then save the six model inputs."""
    # import the large language-model package only when we actually need embeddings.
    # the check-only option can therefore inspect cleaning and splits without downloading MiniLM.
    from sentence_transformers import SentenceTransformer

    # this uses MiniLM's existing knowledge; it does not train or fine-tune MiniLM.
    model = SentenceTransformer("all-MiniLM-L6-v2")
    arrays = {}
    for name, (texts, labels) in splits.items():
        # use the same direct encode call for all three groups; there is no chunking here.
        # long reviews are truncated according to the model's default input limit.
        features = model.encode(texts.tolist(), show_progress_bar=True)
        if features.shape != (len(labels), 384) or not np.isfinite(features).all():
            raise ValueError(f"Unexpected embedding shape or values for {name}.")
        arrays[f"X_{name}"] = features
        arrays[f"y_{name}"] = labels.to_numpy()
        print(f"{name} embeddings: {features.shape}; labels: {labels.shape}")
    # keep row order intact: the nth embedding belongs to the nth rating in the same split.
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    np.savez_compressed(output_path, **arrays)
    print(f"Saved private embeddings to {output_path}")




In [ ]:
# generate one 384-number representation per review and save its matching rating.
# preserve any earlier export unless you deliberately change the overwrite setting.
output_path = Path('/content/embeddings_split.npz')
overwrite_existing = False
if output_path.exists() and not overwrite_existing:
    raise FileExistsError('An export already exists. Download it first, then intentionally set overwrite_existing=True if needed.')
save_embeddings(splits, output_path)

## 6. Download for VS Code

In [ ]:
# download the private export; back up your existing local input before replacing it.
files.download(str(output_path))

## Handoff

The export contains `X_train`, `X_val`, `X_test`, `y_train`, `y_val`, and `y_test`. Each embedding array has 384 columns. Put the intended export in the local project's `data/embeddings_split.npz`, then use `scripts/difficulty_model.py` for training and evaluation. Downloading this notebook as `.py` includes notebook commands; use the companion `scripts/prepare_review_embeddings.py` for ordinary Python execution instead.

The existing selected model uses alpha 0.3 and has validation MAE 0.7433 and test MAE 0.7461. This preparation notebook does not recalculate those scores. Do not commit raw-review outputs or private exports. Before sharing the notebook, use **Edit → Clear all outputs**.